In [1]:
from pathlib import Path
import sys

# Make the local src-layout package importable even if this kernel was already running.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the COE549 project root from the notebook working directory.")
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pydantic import BaseModel
from typing import Optional # Optional is a type hint that allows for a variable to be either a specific type or None.
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage

from llms_course.utils.llms import get_llm

/Users/annamaria/Documents/Fall2026/LLMs/Samers/COE549/.venv/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
class State(BaseModel):
    topic: str # This is required by the graph
    joke_setup: Optional[str] = None # So now we do not have to pass it to .invoke() since it is an Optional type
    joke_punchlines: Optional[str] = None # So now we do not have to pass it to .invoke() since it is an Optional type

In [3]:
def generate_joke_setup(state: State) -> State:
    """
    Generate a joke setup for a given topic using an LLM.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    llm = get_llm()

    system_prompt = """# Role
You are a helpful assistant that generates the setup for jokes based on a topic.

# Instructions
- Output the setup for a joke about the given topic.
- Do not include any punchlines.

# Output Format
- Output the joke setup only without any other text or formatting.
"""

    messages = [
        SystemMessage(content = system_prompt),
        HumanMessage(content = f"Generate a joke setup for a joke about ```{state.topic}```.")
    ]

    response = llm.invoke(messages)
    joke_setup = response.content
    print(joke_setup)
    state.joke_setup = joke_setup
    return state

def generate_joke_punchlines(state: State) -> State:
    """
    Generate three variations of a joke punchline for a given joke setup using an LLM.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    llm = get_llm()
    
    system_prompt = """# Role
You are a helpful assistant that generates the punchlines for jokes based on a setup.

# Instructions
- Output three different punchlines for the given joke setup.

# Output Format
- Output the three punchlines only without any other text or formatting in a numbered list, e.g.
    1. Punchline 1
    2. Punchline 2
    3. Punchline 3
"""

    messages = [
        SystemMessage(content = system_prompt),
        HumanMessage(content = f"Generate three punchlines for the joke setup ```{state.joke_setup}```.")
    ]

    response = llm.invoke(messages)
    punchlines = response.content
    print(punchlines)
    state.joke_punchlines = punchlines
    return state


In [4]:
# Create the graph builder
graph_builder = StateGraph(State)

# Add the nodes to the graph
graph_builder.add_node("generate_joke_setup", generate_joke_setup)
graph_builder.add_node("generate_joke_punchlines", generate_joke_punchlines)

# Add the edges to the graph
graph_builder.add_edge("generate_joke_setup", "generate_joke_punchlines") # From generate_joke_setup -> generate_joke_punchlines
graph_builder.add_edge("generate_joke_punchlines", END) # From generate_joke_punchlines -> END

# Set the entry point to the `generate_joke_setup` node
graph_builder.set_entry_point("generate_joke_setup")

# Compile the graph
graph = graph_builder.compile()

In [5]:
# Input into the graph must match the state schema
inputs = {"topic": "Driving in Lebanon"}

# Invoke the graph with the inputs and return the final state
final_state = graph.invoke(inputs)

print(final_state)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}